# 12 — Qwen3.5-2B on a free Colab GPU: the free "go / no-go" test

**In one sentence:** this notebook asks a small 2026 model to solve easy and medium code
problems, with its *thinking* switched ON and OFF, and tells us — **for free** — whether the plan
works before any money is spent (DECISIONS #59).

**Why this notebook exists.** Everything used to run on a Mac. One medium problem took
**250 seconds**. A free Colab T4 answers 16 problems at the same time, so it is much faster.
Qwen3.5-2B needs about 4.5 GB and the T4 has about 15 GB, so the whole model fits on the GPU.
(Our old model did not fit, and that is why it crawled at 4.4 tokens/s — `DECISIONS #43, #46`.)

**It answers 4 questions:**

| # | Question | Step |
|---|---|---|
| 1 | Does the T4's number format (float16) give garbage? | 6b |
| 2 | Is the model good enough, on easy **and** medium problems? | 7–9 |
| 3 | Is there room to shorten its thinking? | 9 |
| 4 | How fast is it — and would vLLM be faster? | 7, 11 |

**Where we are:** `PROBLEM ✅ → GAP ✅ → QUESTION ✅ → HYPOTHESIS ✅ → EXPERIMENT ⬅ HERE`

**Before you run anything:** menu **Runtime → Change runtime type → T4 GPU**.

Run the cells in order. Steps 3, 6 and 6b are **safety nets** — if they fail, stop. Do not skip them.

## 1. Install what we need
**Problem:** Colab does not have our libraries. **Why:** without them nothing runs.
**In:** nothing. **Out:** installed packages. **Why this way:** plain `transformers` for asking
questions (we only add Unsloth later, for training), plus `evalplus` which holds the HumanEval
problems *and their tests*.

In [ ]:
%%capture
!pip install -q -U "transformers>=5.5.0" accelerate
!pip install -q evalplus

In [ ]:
# Check it worked, and that we really have a GPU. If this says "No GPU", fix the runtime type.
import torch, transformers
print("transformers", transformers.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU  <-- STOP")
print("bfloat16 supported:", torch.cuda.is_bf16_supported() if torch.cuda.is_available() else "-")
print("GPU memory:", f"{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB"
      if torch.cuda.is_available() else "-")

## 2. Get the code, and a safe place to save answers
**Problem:** a free session can die at any second, and then a finished answer is lost.
**Why:** answers cost GPU time; losing one means paying for it twice.
**In:** our public GitHub repo. **Out:** the code, and a results folder on Google Drive.
**Why this way:** Drive outlives the session. The results file is **append-only** and we
`fsync` after every batch, so a crash can never cost more than the batch being written.

In [ ]:
import os, sys

REPO = "https://github.com/mahmudulhaquequdrati/stop-overthinking-thesis.git"
if not os.path.isdir("/content/thesis"):
    !git clone -q {REPO} /content/thesis
else:
    !cd /content/thesis && git pull -q
os.chdir("/content/thesis")
sys.path.insert(0, "/content/thesis/scripts")

try:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = "/content/drive/MyDrive/stop-overthinking/results"
except Exception as e:
    print("No Google Drive (fine when testing locally):", e)
    OUT_DIR = "results"
os.makedirs(OUT_DIR, exist_ok=True)
print("code in :", os.getcwd())
print("saving to:", OUT_DIR)

## 3. ⚠️ SAFETY NET — is the thinking counter correct?
**Problem:** Qwen and our old model mark their thinking differently.

```
OLD (Gemma)  prompt ends: ...            output: <|channel>thought ... <channel|> ANSWER
                                                 ^^^^^^ the START marker is in the OUTPUT

NEW (Qwen)   prompt ends: ... <think>    output: thinking ... </think> ANSWER
                              ^^^^^^^ the START marker is in the PROMPT, not the output
```

Code that looks for the start marker finds **nothing** with Qwen. It would report **0 thinking
tokens on every answer** — and nothing would crash. Thinking tokens are the number this whole
thesis is about, so that mistake would quietly ruin everything.

**In:** nothing (only the tokenizer, a few MB — no GPU needed). **Out:** PASS or FAIL.
**Why this way:** it tests the **real** chat template downloaded from Hugging Face, not a copy
we typed by hand. **If anything says FAIL, stop here.**

In [ ]:
!python scripts/test_prompts.py

## 4. Load the model
**Problem:** we need the model on the GPU. **Why:** everything else waits on this.
**In:** `unsloth/Qwen3.5-2B` (4.58 GB). **Out:** a model and a tokenizer.
**Why this way:** 16-bit, straight onto the GPU, **no CPU offloading** — offloading is exactly
what made our old model 12× slower. A T4 has no `bfloat16`, so the code falls back to `float16`.

👉 **Check the printed memory: it should be about 5 GB.** If it is near 14 GB, something loaded
in the wrong precision — stop and ask.

In [ ]:
import models, prompts
from gen_colab import load_model

profile = models.get("qwen35_2b")
model, tok = load_model(profile)

# Show that the switch really changes the prompt.
on, off = prompts.check_prompt_has_switch(tok, profile)
print("\nthinking ON  prompt ends:", repr(on[-40:]))
print("thinking OFF prompt ends:", repr(off[-40:]))

## 5. Get the problems
**Problem:** every run used to download the benchmarks again. **Why:** slow, and it needs
internet mid-run. **In:** HumanEval (easy) + LiveCodeBench (easy and medium).
**Out:** one local file `data/problems.json` with the questions *and* their tests.
**Why this way:** written once, only read afterwards, so a run can never change the problems
under our feet.

In [ ]:
!python scripts/build_problem_set.py

## 6. ⚠️ SAFETY NET — a 2-problem smoke test
**Problem:** a 2-hour run that was wrong from the first second is the most expensive mistake
we can make. **Why:** we check the two things that silently break, on 2 problems, in ~1 minute.
**In:** 2 problems. **Out:** two token counts.

👉 **What you must see:**
- `thinking_on` → thinking tokens in the **hundreds**. If it prints **0**, the trap from cell 3
  is still there. **Stop.**
- `thinking_off` → thinking tokens **exactly 0** on every row. If not, the switch is being ignored.

In [ ]:
SMOKE = f"{OUT_DIR}/smoke-test.jsonl"
!rm -f "{SMOKE}"
!python scripts/gen_colab.py --policy thinking_on  --n 2 --batch 2 --max-tokens 1024 --out "{SMOKE}"
!python scripts/gen_colab.py --policy thinking_off --n 2 --batch 2 --max-tokens 1024 --out "{SMOKE}"

In [ ]:
import json
print(f"{'policy':<14}{'thinking':>10}{'total':>8}   first 60 characters of the answer")
for line in open(SMOKE):
    r = json.loads(line)
    print(f"{r['policy']:<14}{r['thinking_tokens']:>10}{r['total_new_tokens']:>8}   "
          f"{r['answer_text'][:60].strip()!r}")

rows = [json.loads(l) for l in open(SMOKE)]
on  = [r for r in rows if r["policy"] == "thinking_on"]
off = [r for r in rows if r["policy"] == "thinking_off"]
ok = all(r["thinking_tokens"] > 0 for r in on) and all(r["thinking_tokens"] == 0 for r in off)
print("\n" + ("SMOKE TEST PASSED - go on to the pilot." if ok else
               "SMOKE TEST FAILED - do NOT run the pilot. Read cell 3 again."))

## 6b. ⚠️ SAFETY NET — does float16 give garbage?
**Problem:** Qwen was trained in a number format called **bfloat16**. The T4 is too old for it,
so it uses **float16**, a close cousin that can "overflow" when numbers get very big. Then the
model writes garbage (often `!!!!!!` or nothing at all). **Why:** if that happens, every answer
in this notebook is wrong, and so is every decision we make from it.

**In:** 10 easy problems, thinking ON (long answers are the hardest test), answered twice —
once in `float16`, once in `float32` (the big, safe format, about 2× slower).
**Out:** pass counts for both, a garbage count, and a verdict.
**Why this way:** `float32` cannot overflow, so it is the honest reference. The rule is fixed
**before** we look (DECISIONS #59):

```
float16 is OK  if  it passes at most 1 problem fewer than float32
              AND  it has no more garbage answers than float32
otherwise      ->  every run below uses float32, 8 at a time (it needs ~9 GB for the model)
```

It takes about 5–10 minutes. It is free insurance for the next two hours.

In [ ]:
# The scripts below load their OWN copy of the model. Free the one from cell 4 first,
# or float32 (~9 GB) + this copy (~5 GB) would not fit in the T4's 15 GB.
import gc, torch
globals().pop("model", None)
gc.collect(); torch.cuda.empty_cache()
print(f"GPU memory still used by this notebook: {torch.cuda.memory_allocated()/1e9:.2f} GB (should be ~0)")

FP = {d: f"{OUT_DIR}/2026-09-21-fp-check-{d}.jsonl" for d in ("float16", "float32")}
!python scripts/gen_colab.py --policy thinking_on --n 10 --batch 10 --dtype float16 --max-tokens 4096 --out "{FP['float16']}"
!python scripts/gen_colab.py --policy thinking_on --n 10 --batch 5  --dtype float32 --max-tokens 4096 --out "{FP['float32']}"
!python scripts/grade_humaneval.py --answers "{FP['float16']}" > /dev/null
!python scripts/grade_humaneval.py --answers "{FP['float32']}" > /dev/null

In [ ]:
import csv, json, re

# Garbage = one character 50+ times in a row (what float16 overflow usually looks like; spaces
# and divider lines like ----- are allowed), or an answer that FINISHED thinking and still gave
# no code block. float32 is scored the same way, so a strict check hurts both equally.
REPEAT = re.compile(r"([^\s\-=#*_])\1{49,}")

def summary(dtype):
    graded = list(csv.DictReader(open(FP[dtype].replace(".jsonl", "-graded.csv"))))
    rows = [json.loads(l) for l in open(FP[dtype])]
    garbage = sum(bool(REPEAT.search(r["raw_output"])) or
                  (not r["hit_limit"] and "```" not in r["answer_text"]) for r in rows)
    # each answer carries its share of its batch's time, so the seconds add up exactly
    secs = sum(r["batch_seconds"] / r["batch_size"] for r in rows)
    return sum(g["passed"] == "True" for g in graded), len(graded), garbage, \
           sum(r["total_new_tokens"] for r in rows) / secs

s16, s32 = summary("float16"), summary("float32")
for name, (passed, n, garbage, tps) in (("float16", s16), ("float32", s32)):
    print(f"{name}: passed {passed}/{n} | garbage answers {garbage} | ~{tps:.0f} tokens/s")

FP16_OK = s16[0] >= s32[0] - 1 and s16[2] <= s32[2]
DTYPE, BATCH = ("float16", 16) if FP16_OK else ("float32", 8)
print("\n" + ("float16 is SAFE -> the pilot uses float16, 16 at a time." if FP16_OK else
              "float16 is NOT safe -> the pilot uses float32, 8 at a time. Slower, still free."))

## 7. The pilot — the run that decides everything
**Problem:** is Qwen3.5-2B good enough to carry the thesis, on easy **and** medium problems?
**Why:** we must know before we spend days on it — and before we spend any money (DECISIONS #59).
**In:** 30 easy problems (HumanEval) + 20 **medium** problems (LiveCodeBench), **4 tries each**,
both ways of answering. **Out:** 400 answers saved to Drive, in two files.
**Why this way:** medium problems are where a 2B model is most likely to fail, so a test
without them would tell us nothing we need. 4 tries is the smallest number that can show
whether *some* correct answers are shorter than others — which is the whole point.

Every way of answering gets the **same** token limit (4,096) and the same settings. Only the
switch changes. That is the fairness rule in `CLAUDE.md` §4. The number format (`DTYPE`)
comes from step 6b.

**If Colab disconnects:** run steps 1, 2, 5 and 6b again (6b skips the answers it already has),
then this cell. It also skips what is already finished.
Estimated time: about 1–3 hours (not measured yet — the printed tokens/s will tell us).

In [ ]:
PILOT     = f"{OUT_DIR}/2026-09-21-qwen-pilot.jsonl"          # easy: HumanEval
PILOT_LCB = f"{OUT_DIR}/2026-09-21-qwen-pilot-lcb.jsonl"      # medium: LiveCodeBench
for policy in ("thinking_off", "thinking_on"):
    !python scripts/gen_colab.py --policy {policy} --source humaneval --n 30 --samples 4 \
        --dtype {DTYPE} --batch {BATCH} --max-tokens 4096 --out "{PILOT}"
    !python scripts/gen_colab.py --policy {policy} --source lcb --difficulty medium --n 20 --samples 4 \
        --dtype {DTYPE} --batch {BATCH} --max-tokens 4096 --out "{PILOT_LCB}"

## 8. Grade with the benchmark's own tests
**Problem:** we must never judge an answer by eye. **Why:** that is how people fool themselves.
**In:** the answers. **Out:** a `-graded.csv`, one row per answer.
**Why this way:** each answer runs in its **own short-lived process** with a time limit, never
inside this notebook (`CLAUDE.md` §4). The tests come from the benchmark itself.

The cell first grades HumanEval's **official solutions**. They must score about 100%. If they do
not, the grader is broken and every accuracy number below would be meaningless.

In [ ]:
# Sanity check on the grader itself, using the benchmark's own correct answers.
import json, tempfile
from evalplus.data import get_human_eval_plus
import grade_humaneval as G

problems = get_human_eval_plus()
with tempfile.TemporaryDirectory() as wd:
    checked = list(problems.items())[:10]
    passed = sum(G.run_one(p["prompt"] + p["canonical_solution"], p, wd)[0] for _, p in checked)
print(f"grader sanity check: {passed}/{len(checked)} official solutions pass "
      f"{'OK' if passed == len(checked) else '<-- GRADER IS BROKEN, STOP'}")

In [ ]:
!python scripts/grade_humaneval.py --answers "{PILOT}"
!python scripts/grade_lcb.py --answers "{PILOT_LCB}"

## 9. The three gates
**Problem:** we must decide with numbers we fixed in advance, not with a feeling.
**Why:** otherwise we will talk ourselves into whatever the result happens to be.
**In:** the graded file. **Out:** three numbers, each PASS or FAIL.

| Gate | Must be | In plain words |
|---|---|---|
| Solved at least once | **≥ 40%** | is the model good enough to have anything to learn from? |
| Room to shorten | **≤ 0.75** | **your "will fine-tuning cut tokens?" question** — is there at least 25% to cut? |
| Cost | tokens per answer | can we finish inside free Colab? |

In [ ]:
# Checked SEPARATELY: a model can pass easily on easy problems and fail on medium ones.
for label, path in (("EASY - HumanEval", PILOT), ("MEDIUM - LiveCodeBench", PILOT_LCB)):
    graded = path.replace(".jsonl", "-graded.csv")
    print("=" * 70, "\n", label)
    !python scripts/check_gates.py --graded "{graded}" --policy thinking_on

## 10. You are here — what the numbers mean

**Decided before we looked** (DECISIONS #59), so we cannot fool ourselves. Everything above
was **free**. Money is spent only in the last line, and only after everything else passed.

```
float16 gave garbage       -> already handled in 6b: everything used float32. Still free.
2B solved < 40%            -> change to Qwen3.5-4B (--model qwen35_4b), run this notebook again. Free.
4B also solved < 40%       -> STOP. Rethink the scope (e.g. easy problems only). $0 spent.
room to shorten > 0.75     -> do NOT change model yet. Run 8 tries instead of 4 (DECISIONS #27).
                              If it still fails, that is a REAL FINDING. Write it down honestly.
all pass, T4 fast enough   -> stay on the free T4 for the whole thesis.
all pass, T4 too slow      -> NOW buy the $10 package (DECISIONS #48): an L4, plus vLLM if
                              step 11 says it runs. We pay knowing the plan works.
```

"Fast enough" = the full plan fits in the week. Rough rule: at the tokens/s printed above,
~15M tokens (testing + training data) must take **under ~20 T4 hours**, i.e. about **200+ tokens/s**.

**Write the numbers down** in `results/2026-09-21-qwen-pilot.md` and add the `DECISIONS.md`
rows. A number that is not written down did not happen.

## 11. (Optional, last) Does vLLM run here, and how fast?
**Problem:** plain `transformers` may be too slow for one week. vLLM is a tool built only
for writing answers fast. **Why:** if the pilot is too slow, we need to know whether vLLM would
fix it **before** we pay for a bigger GPU because of it.
**In:** 16 easy problems, thinking ON, same settings and token limit.
**Out:** "vLLM RUNS" + tokens/s, or "vLLM FAILS" + the error. Both answers are useful.
**Why this way:** it is **last**, because installing vLLM may change the `transformers`
version the pilot used. These answers are only a speed test — they are **not** thesis results.

👉 **Before running:** menu **Runtime → Restart session**. Then run cell 2 (code + Drive) and
this cell. You do NOT need to re-run the pilot.

In [ ]:
!pip install -q vllm
!python scripts/try_vllm.py --n 16 --out "{OUT_DIR}/2026-09-21-vllm-speed.jsonl"